# 🚀 Phase 1: 4D Gaussian Splatting — Baseline Verification

This notebook verifies the base 4D-GS dynamic reconstruction pipeline by training on a D-NeRF synthetic scene and evaluating novel view synthesis quality.

**What this notebook does:**
1. Sets up the GPU environment and compiles CUDA extensions
2. Downloads the D-NeRF `bouncingballs` benchmark scene
3. Trains 4D-GS (3,000 coarse + 20,000 fine iterations)
4. Renders test views and a 360° novel-view video
5. Computes PSNR, SSIM, LPIPS metrics
6. Displays results inline for visual verification

> **Runtime**: ~15-25 minutes on T4, ~8-12 minutes on A100

In [ ]:
# Cell 1: Verify GPU availability
import torch
import subprocess

print("=" * 50)
print("GPU Environment Check")
print("=" * 50)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✅ GPU: {gpu_name}")
    print(f"   Memory: {gpu_mem:.1f} GB")
    print(f"   CUDA: {torch.version.cuda}")
    print(f"   PyTorch: {torch.__version__}")
else:
    print("❌ No GPU detected!")
    print("   Go to Runtime → Change runtime type → GPU")
    raise RuntimeError("GPU required for 4D Gaussian Splatting")

print()
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)

In [ ]:
# Cell 2: Clone repository and install dependencies
import os

# Clone 4DGaussians repo
if not os.path.exists("/content/4DGaussians"):
    !git clone https://github.com/hustvl/4DGaussians.git /content/4DGaussians
    %cd /content/4DGaussians
    !git submodule update --init --recursive
else:
    %cd /content/4DGaussians
    print("Repository already cloned.")

# Install dependencies
print("\n" + "=" * 50)
print("Installing dependencies...")
print("=" * 50)

!pip install -q plyfile open3d imageio[ffmpeg] lpips pytorch_msssim mmcv==1.6.0 matplotlib tqdm

# Compile CUDA extensions
print("\nCompiling CUDA rasterization extension...")
!pip install -e submodules/depth-diff-gaussian-rasterization/
print("\nCompiling simple-knn extension...")
!pip install -e submodules/simple-knn/

print("\n✅ Environment setup complete!")

## 📦 Dataset Download
Downloading the D-NeRF `bouncingballs` synthetic scene (~150 MB).
This scene features three bouncing balls with dynamic motion — a standard benchmark for temporal 3D reconstruction.

In [ ]:
# Cell 3: Download D-NeRF bouncingballs dataset
import os
import zipfile

os.makedirs("/content/4DGaussians/data/dnerf", exist_ok=True)

# Download from HuggingFace (same source as original notebook)
if not os.path.exists("/content/4DGaussians/data/dnerf/bouncingballs"):
    print("Downloading D-NeRF dataset...")
    !wget -q --show-progress \
        https://huggingface.co/camenduru/4DGaussians/resolve/main/data/data.zip \
        -O /content/data.zip
    
    print("Extracting...")
    with zipfile.ZipFile("/content/data.zip", "r") as z:
        z.extractall("/content/4DGaussians/data/")
    
    # The archive extracts with a nested structure — find and move if needed
    import shutil
    # Check various possible extraction paths
    for candidate in ["/content/4DGaussians/data/data/bouncingballs",
                       "/content/4DGaussians/data/bouncingballs"]:
        if os.path.exists(candidate + "/transforms_train.json"):
            if candidate != "/content/4DGaussians/data/dnerf/bouncingballs":
                if os.path.exists("/content/4DGaussians/data/dnerf/bouncingballs"):
                    shutil.rmtree("/content/4DGaussians/data/dnerf/bouncingballs")
                shutil.move(candidate, "/content/4DGaussians/data/dnerf/bouncingballs")
            break
    
    os.remove("/content/data.zip")
    print("✅ Dataset downloaded and extracted!")
else:
    print("Dataset already exists.")

# Verify dataset structure
import json

dataset_path = "/content/4DGaussians/data/dnerf/bouncingballs"
assert os.path.exists(f"{dataset_path}/transforms_train.json"), "Missing transforms_train.json!"
assert os.path.exists(f"{dataset_path}/transforms_test.json"), "Missing transforms_test.json!"

with open(f"{dataset_path}/transforms_train.json") as f:
    train_meta = json.load(f)
with open(f"{dataset_path}/transforms_test.json") as f:
    test_meta = json.load(f)

print(f"\nDataset: bouncingballs")
print(f"  Training frames: {len(train_meta['frames'])}")
print(f"  Test frames: {len(test_meta['frames'])}")
print(f"  Camera FOV: {train_meta.get('camera_angle_x', 'N/A'):.4f} rad")
print(f"  Time range: {train_meta['frames'][0].get('time', 0)} → {train_meta['frames'][-1].get('time', 'N/A')}")

# Show a sample frame
from PIL import Image
import matplotlib.pyplot as plt

sample_path = os.path.join(dataset_path, train_meta['frames'][0]['file_path'].lstrip('./') + '.png')
if os.path.exists(sample_path):
    img = Image.open(sample_path)
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.title(f"Sample frame: {os.path.basename(sample_path)}")
    plt.axis('off')
    plt.show()
    print(f"  Image size: {img.size}")

## 🏋️ Training
Running the full 4D-GS training pipeline:
- **Stage 1 (Coarse)**: 3,000 iterations — static 3D Gaussians, no temporal deformation
- **Stage 2 (Fine)**: 20,000 iterations — HexPlane temporal field + deformation MLPs enabled

In [ ]:
# Cell 4: Train 4D-GS on bouncingballs
import time

%cd /content/4DGaussians

scene = "bouncingballs"
exp_name = f"dnerf/{scene}"
data_path = f"data/dnerf/{scene}"
config_path = f"arguments/dnerf/{scene}.py"

print("=" * 60)
print(f"  Training 4D Gaussian Splatting on: {scene}")
print(f"  Config: {config_path}")
print(f"  Coarse iters: 3,000 | Fine iters: 20,000")
print("=" * 60)

start_time = time.time()

!python train.py \
    -s {data_path} \
    --port 6017 \
    --expname {exp_name} \
    --configs {config_path}

elapsed = time.time() - start_time
print(f"\n✅ Training complete in {elapsed/60:.1f} minutes")
print(f"   Output saved to: output/{exp_name}/")

## 🎬 Rendering
Generating:
- Test-set novel view renderings (for quantitative evaluation)
- 360° orbit video with continuous temporal interpolation

In [ ]:
# Cell 5: Render test views and novel-view video
%cd /content/4DGaussians

scene = "bouncingballs"
exp_name = f"dnerf/{scene}"
model_path = f"output/{exp_name}"
config_path = f"arguments/dnerf/{scene}.py"

print("=" * 60)
print(f"  Rendering test views and novel-view video")
print("=" * 60)

!python render.py \
    --model_path {model_path} \
    --skip_train \
    --configs {config_path}

print("\n✅ Rendering complete!")

# List output files
import os
for root, dirs, files in os.walk(f"{model_path}/test"):
    for f in sorted(files)[:5]:
        print(f"  {os.path.join(root, f)}")
for root, dirs, files in os.walk(f"{model_path}/video"):
    for f in sorted(files)[:5]:
        print(f"  {os.path.join(root, f)}")

## 📊 Metrics
Computing quantitative evaluation metrics on the test set:
- **PSNR**: Peak Signal-to-Noise Ratio (higher is better, target ≥ 30 dB)
- **SSIM**: Structural Similarity Index (higher is better)
- **LPIPS**: Learned Perceptual Image Patch Similarity (lower is better)

In [ ]:
# Cell 6: Compute evaluation metrics
%cd /content/4DGaussians

scene = "bouncingballs"
model_path = f"output/dnerf/{scene}"

print("=" * 60)
print(f"  Computing PSNR / SSIM / LPIPS on test set")
print("=" * 60)

!python metrics.py --model_path {model_path}

# Read and display results
import json
import os

results_path = os.path.join(model_path, "results.json")
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    print("\n" + "=" * 40)
    print("  EVALUATION RESULTS")
    print("=" * 40)
    for iteration, metrics in results.items():
        print(f"\n  Iteration: {iteration}")
        for metric_name, value in metrics.items():
            if isinstance(value, float):
                print(f"    {metric_name}: {value:.4f}")
            else:
                print(f"    {metric_name}: {value}")
    
    # Quick pass/fail check
    last_iter = list(results.keys())[-1]
    psnr = results[last_iter].get("PSNR", 0)
    if psnr >= 30.0:
        print(f"\n  ✅ PSNR = {psnr:.2f} dB — BASELINE VERIFIED!")
    else:
        print(f"\n  ⚠️ PSNR = {psnr:.2f} dB — below 30 dB target")
        print(f"     This may be normal for some scenes. Check renderings visually.")
else:
    print("\n  ⚠️ results.json not found. Check metrics.py output above.")

## 🎥 Visualization
Displaying the rendered novel-view video and GT comparisons.

In [ ]:
# Cell 7: Display rendered video and image comparisons
import os
import glob
from IPython.display import HTML, display
from base64 import b64encode
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

scene = "bouncingballs"
model_path = f"output/dnerf/{scene}"

# --- Display Novel-View Video ---
print("=" * 50)
print("  Novel-View 360° Video")
print("=" * 50)

# Find the video file
video_files = glob.glob(f"{model_path}/video/**/video_rgb.mp4", recursive=True)
if video_files:
    video_path = video_files[0]
    with open(video_path, "rb") as f:
        video_data = f.read()
    b64_video = b64encode(video_data).decode()
    html = f'''
    <video width="600" controls autoplay loop>
        <source src="data:video/mp4;base64,{b64_video}" type="video/mp4">
    </video>
    <p style="text-align:center; color:gray">360° novel-view synthesis with temporal dynamics</p>
    '''
    display(HTML(html))
    print(f"  Video: {video_path}")
else:
    print("  ⚠️ Novel-view video not found.")

# --- Side-by-Side Comparison: Rendered vs GT ---
print("\n" + "=" * 50)
print("  Test Set: Rendered vs Ground Truth")
print("=" * 50)

# Find render and GT directories
render_dirs = glob.glob(f"{model_path}/test/**/renders", recursive=True)
gt_dirs = glob.glob(f"{model_path}/test/**/gt", recursive=True)

if render_dirs and gt_dirs:
    render_dir = render_dirs[0]
    gt_dir = gt_dirs[0]
    
    render_files = sorted(glob.glob(f"{render_dir}/*.png"))
    gt_files = sorted(glob.glob(f"{gt_dir}/*.png"))
    
    # Show 4 evenly spaced comparisons
    n_show = min(4, len(render_files))
    indices = np.linspace(0, len(render_files) - 1, n_show, dtype=int)
    
    fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))
    if n_show == 1:
        axes = axes[:, np.newaxis]
    
    for col, idx in enumerate(indices):
        # Ground truth
        gt_img = Image.open(gt_files[idx])
        axes[0, col].imshow(gt_img)
        axes[0, col].set_title(f"GT #{idx}", fontsize=10)
        axes[0, col].axis("off")
        
        # Rendered
        render_img = Image.open(render_files[idx])
        axes[1, col].imshow(render_img)
        axes[1, col].set_title(f"Rendered #{idx}", fontsize=10)
        axes[1, col].axis("off")
    
    axes[0, 0].set_ylabel("Ground Truth", fontsize=12)
    axes[1, 0].set_ylabel("4D-GS Rendered", fontsize=12)
    plt.suptitle("Test Set Comparison", fontsize=14)
    plt.tight_layout()
    plt.show()
    print(f"  Showing {n_show} of {len(render_files)} test frames.")
else:
    print("  ⚠️ Render/GT directories not found.")

print("\n" + "=" * 50)
print("  ✅ Phase 1 Complete — Baseline Verified!")
print("  Next: Run Phase 2 notebook for semantic extraction")
print("=" * 50)